# Product Data Cleaning

## Objective

Clean and validate product master data before loading into the analytics database.

## Cleaning Tasks

- Identify missing product attributes
- Remove duplicate product IDs
- Standardize product naming
- Validate SKU formats
- Check pricing consistency
- Export cleaned product data

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
raw_path = Path("../data/raw/products.csv")

clean_path = Path("../data/cleaned/products_clean.csv")

In [3]:
products = pd.read_csv(raw_path)

products.head()

,product_id,sku,brand,category,product_name,flavor,package_size,units_per_case,case_cost,case_price,active
0,1001,MONS-ORI,Monster Energy,Energy Drink,Monster Energy Original,Original,16 oz Can,24,22.58,35.50,True
1,1002,MONS-ZERSUG,Monster Energy,Energy Drink,Monster Energy Zero Sugar,Zero Sugar,16 oz Can,24,22.95,31.22,True
2,1003,MONS-LO-,Monster Energy,Energy Drink,Monster Energy Lo-Carb,Lo-Carb,16 oz Can,24,24.18,33.60,True
3,1004,MONS-IMP,Monster Energy,Energy Drink,Monster Energy Import,Import,18.6 oz Can,12,26.03,34.43,True
4,1005,MONS-ULTWHI,Monster Ultra,Ultra,Monster Ultra Ultra White,Ultra White,16 oz Can,24,27.70,36.69,True


In [4]:
products.shape

(34, 11)

In [5]:
products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34 entries, 0 to 33
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   product_id      34 non-null     int64  
 1   sku             34 non-null     object 
 2   brand           34 non-null     object 
 3   category        34 non-null     object 
 4   product_name    34 non-null     object 
 5   flavor          33 non-null     object 
 6   package_size    34 non-null     object 
 7   units_per_case  34 non-null     int64  
 8   case_cost       34 non-null     float64
 9   case_price      34 non-null     float64
 10  active          34 non-null     bool   
dtypes: bool(1), float64(2), int64(2), object(6)
memory usage: 2.8+ KB


In [6]:
products.isnull().sum()

product_id        0
sku               0
brand             0
category          0
product_name      0
flavor            1
package_size      0
units_per_case    0
case_cost         0
case_price        0
active            0
dtype: int64

In [7]:
products["product_id"].duplicated().sum()

1

In [8]:
products["product_name"].duplicated().sum()

0

In [9]:
products[
    products["product_id"].duplicated(keep=False)
]

,product_id,sku,brand,category,product_name,flavor,package_size,units_per_case,case_cost,case_price,active
19,1020,JUIC-AUSSTYLEM,Juice Monster,Juice,Juice Monster Aussie Style Lemonade,Aussie Style Lemonade,16 oz Can,24,27.07,39.76,True
20,1020,JUIC-PAP,Juice Monster,Juice,Juice Monster Papillon,Papillon,16 oz Can,24,29.76,39.42,True


Product Data Quality Findings:

- Product master contained 34 records across 11 attributes.
- 1 missing flavor value was identified.
- 1 duplicate product identifier was detected.
- No duplicate product names were found.

In [10]:
products[
    products["product_id"].duplicated(keep=False)
]

,product_id,sku,brand,category,product_name,flavor,package_size,units_per_case,case_cost,case_price,active
19,1020,JUIC-AUSSTYLEM,Juice Monster,Juice,Juice Monster Aussie Style Lemonade,Aussie Style Lemonade,16 oz Can,24,27.07,39.76,True
20,1020,JUIC-PAP,Juice Monster,Juice,Juice Monster Papillon,Papillon,16 oz Can,24,29.76,39.42,True


In [11]:
products_clean = products.copy()

In [12]:
products_clean["flavor"] = (
    products_clean["flavor"]
    .fillna("Unknown")
)

In [13]:
products_clean["product_id"].max()

1034

In [14]:
products_clean.loc[
    products_clean["product_id"].duplicated(),
    "product_id"
] = products_clean["product_id"].max() + 1

In [15]:
products_clean["product_id"].duplicated().sum()

0

In [16]:
products_clean.isnull().sum()

product_id        0
sku               0
brand             0
category          0
product_name      0
flavor            0
package_size      0
units_per_case    0
case_cost         0
case_price        0
active            0
dtype: int64

In [17]:
products_clean["sku"].duplicated().sum()


0

In [18]:
products_clean["sku"].head(10)

0       MONS-ORI
1    MONS-ZERSUG
2       MONS-LO-
3       MONS-IMP
4    MONS-ULTWHI
5    MONS-ULTBLU
6    MONS-ULTRED
7    MONS-ULTPAR
8    MONS-ULTSUN
9    MONS-ULTVIO
Name: sku, dtype: object

In [19]:
products_clean[
    products_clean["case_price"] <= products_clean["case_cost"]
]

,product_id,sku,brand,category,product_name,flavor,package_size,units_per_case,case_cost,case_price,active
25,1026,JAVA-IRICRÈ,Java Monster,Coffee Energy,Java Monster Irish Crème,Irish Crème,15 oz Can,12,24.84,-30.88,True


In [20]:
products_clean[
    products_clean["brand"] == "Java Monster"
][["product_name", "case_cost", "case_price"]]

,product_name,case_cost,case_price
23,Java Monster Mean Bean,20.56,35.71
24,Java Monster Loca Moca,21.41,31.86
25,Java Monster Irish Crème,24.84,-30.88
26,Java Monster French Vanilla,20.28,35.90
27,Java Monster Salted Caramel,23.67,34.61


In [21]:
products_clean["case_price"].describe()

count    34.000000
mean     34.478235
std      11.763442
min     -30.880000
25%      34.682500
50%      36.410000
75%      38.552500
max      39.760000
Name: case_price, dtype: float64

Product Data Quality Findings:

- Identified missing flavor attributes.
- Resolved duplicate product identifiers.
- Validated SKU uniqueness.
- Detected and corrected invalid pricing values.
- Confirmed product master data integrity.

In [22]:
products_clean.loc[
    products_clean["case_price"] <= 0,
    "case_price"
] = products_clean["case_price"].median()

In [23]:
products_clean[
    products_clean["case_price"] <= 0
]

,product_id,sku,brand,category,product_name,flavor,package_size,units_per_case,case_cost,case_price,active


In [24]:
products_clean["case_price"].describe()

count    34.000000
mean     36.457353
std       2.238444
min      31.220000
25%      34.932500
50%      36.450000
75%      38.552500
max      39.760000
Name: case_price, dtype: float64

In [25]:
products_clean.to_csv(
    clean_path,
    index=False
)

In [1]:
invoices_clean[
    invoices_clean["product_id"] == 1021
]

NameError: name 'invoices_clean' is not defined